In [3]:
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!python3 get-pip.py --user

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.1 MB/s  0:00:00m eta 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import sys
import os

user_site = os.path.expanduser("~/.local/lib/python" + str(sys.version_info.major) + "." + str(sys.version_info.minor) + "/site-packages")
sys.path.append(user_site)

In [5]:
!python3 -m pip install --user playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 18.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.0/612.0 kB 1.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [playwright]  WARNING: The script playwright is installed in '/Commjhub/jupyterhub/home/kylegrg/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [playwright]4 [playwright]


In [6]:
!python3 -m playwright install firefox

(node:4011003) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
99.5 MiB [                    ] 0% 75.7s99.5 MiB [                    ] 0% 5.7s99.5 MiB [                    ] 1% 2.7s99.5 MiB [                    ] 2% 2.6s99.5 MiB [=                   ] 2% 2.6s99.5 MiB [=                   ] 3% 2.7s99.5 MiB [=                   ] 3% 2.8s99.5 MiB [=                   ] 3% 2.9s99.5 MiB [=                   ] 4% 3.0s99.5 MiB [=                   ] 5% 3.0s99.5 MiB [=                   ] 6% 3.1s99.5 MiB [=                   ] 6% 3.0s99.5 MiB [=                   ] 7% 3.1s99.5 MiB [==                  ] 7% 3.0s99.5 MiB [==                  ] 8% 3.0s99.5 MiB [==                  ] 9% 3.0s99.5 MiB [==                  ] 10% 2.9s99.5 MiB [==                  ] 

In [11]:
import asyncio
import json
import re
import argparse
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from playwright.async_api import async_playwright, Response

In [16]:
DEFAULT_HASHTAG  = "coding"
DEFAULT_MAX      = 50
OUTPUT_FILE      = "tiktok_{hashtag}_{ts}.csv"
 
# TikTok API endpoint patterns we care about
TIKTOK_FEED_PATTERNS = [
    "/api/challenge/item_list",   # hashtag feed
    "/api/recommend/item_list",   # sometimes used instead
]
 
 
# ── Data extractor ─────────────────────────────────────────────────────────────
 
def parse_tiktok_item(item: dict) -> dict | None:
    """Flatten a single TikTok video object into a row dict."""
    try:
        author  = item.get("author", {})
        stats   = item.get("stats", {})
        music   = item.get("music", {})
        video   = item.get("video", {})
        desc    = item.get("desc", "")
        challenges = [c.get("title", "") for c in item.get("challenges", [])]
 
        return {
            "post_id":          item.get("id"),
            "description":      desc,
            "hashtags":         " ".join(f"#{t}" for t in challenges),
            "author_username":  author.get("uniqueId"),
            "author_nickname":  author.get("nickname"),
            "author_followers": author.get("stats", {}).get("followerCount"),
            "author_verified":  author.get("verified", False),
            "likes":            stats.get("diggCount"),
            "comments":         stats.get("commentCount"),
            "shares":           stats.get("shareCount"),
            "views":            stats.get("playCount"),
            "music_title":      music.get("title"),
            "music_author":     music.get("authorName"),
            "video_duration":   video.get("duration"),
            "created_at":       datetime.utcfromtimestamp(
                                    item.get("createTime", 0)
                                ).strftime("%Y-%m-%d %H:%M:%S"),
            "post_url":         f"https://www.tiktok.com/@{author.get('uniqueId')}/video/{item.get('id')}",
        }
    except Exception as e:
        print(f"  [!] Skipped item — parse error: {e}")
        return None
 
 
# ── Response handler ───────────────────────────────────────────────────────────
 
def make_response_handler(collected: list, max_posts: int):
    """
    Returns an async handler to attach to `page.on('response')`.
    It fires on every HTTP response; we filter for TikTok feed endpoints.
    """
    async def handler(response: Response):
        if len(collected) >= max_posts:
            return
 
        url = response.url
        if not any(pat in url for pat in TIKTOK_FEED_PATTERNS):
            return
 
        try:
            body = await response.json()
        except Exception:
            return  # Not JSON — skip
 
        items = body.get("itemList") or body.get("items") or []
        before = len(collected)
 
        for item in items:
            if len(collected) >= max_posts:
                break
            row = parse_tiktok_item(item)
            if row and row["post_id"] not in {r["post_id"] for r in collected}:
                collected.append(row)
 
        added = len(collected) - before
        if added:
            print(f"  ✓ Captured {added} posts (total: {len(collected)}/{max_posts})")
 
    return handler
 
 
# ── Main scraper ───────────────────────────────────────────────────────────────
 
async def scrape_tiktok_hashtag(hashtag: str, max_posts: int = DEFAULT_MAX) -> pd.DataFrame:
    hashtag = hashtag.lstrip("#")
    url     = f"https://www.tiktok.com/tag/{hashtag}"
    collected: list[dict] = []
 
    print(f"\n🎵 TikTok Hashtag Scraper")
    print(f"   Hashtag : #{hashtag}")
    print(f"   Target  : {max_posts} posts")
    print(f"   URL     : {url}\n")
 
    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)   # headless=True for CI
        context = await browser.new_context(
            viewport={"width": 1280, "height": 900},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) "
                "Gecko/20100101 Firefox/125.0"
            ),
            locale="en-US",
        )
        page = await context.new_page()
 
        # ── Attach the event listener BEFORE navigation ──────────────────────
        page.on("response", make_response_handler(collected, max_posts))
 
        await page.goto(url, wait_until="domcontentloaded", timeout=30_000)
        await page.wait_for_timeout(3000)   # let initial feed load
 
        # ── Scroll to trigger pagination requests ─────────────────────────────
        scroll_attempts = 0
        max_scrolls     = 40
 
        while len(collected) < max_posts and scroll_attempts < max_scrolls:
            await page.evaluate("window.scrollBy(0, window.innerHeight * 2)")
            await page.wait_for_timeout(1800)   # wait for XHR
            scroll_attempts += 1
 
            if scroll_attempts % 5 == 0:
                print(f"  … scroll #{scroll_attempts}, posts so far: {len(collected)}")
 
        await browser.close()
 
    print(f"\n✅ Done — {len(collected)} posts collected.")
    return pd.DataFrame(collected)
 
 
# ── CLI entry point ────────────────────────────────────────────────────────────
 
# ── Instead of argparse, set your params here directly ──
HASHTAG   = "oscarssowhite"   # ← change this
MAX_POSTS = 500         # ← change this
OUTPUT    = "oscarssowhite_tiktok.csv"       # ← or set a path like "results.csv"

df = await scrape_tiktok_hashtag(HASHTAG, MAX_POSTS)

if df.empty:
    print("⚠️  No posts collected. TikTok may require login or has changed its API.")
else:
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = OUTPUT or OUTPUT_FILE.format(hashtag=HASHTAG, ts=ts)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"💾 Saved → {path}")
    print(df.head())


🎵 TikTok Hashtag Scraper
   Hashtag : #oscarssowhite
   Target  : 500 posts
   URL     : https://www.tiktok.com/tag/oscarssowhite

  … scroll #5, posts so far: 0
  … scroll #10, posts so far: 0
  … scroll #15, posts so far: 0
  … scroll #20, posts so far: 0
  … scroll #25, posts so far: 0
  … scroll #30, posts so far: 0
  … scroll #35, posts so far: 0
  … scroll #40, posts so far: 0

✅ Done — 0 posts collected.
⚠️  No posts collected. TikTok may require login or has changed its API.


In [20]:
# ── DEBUG VERSION — paste this whole cell ──

import asyncio, json, re, pandas as pd
from datetime import datetime
from playwright.async_api import async_playwright

HASHTAG   = "oscarssowhite"
MAX_POSTS = 500

async def scrape_tiktok_debug(hashtag, max_posts=50):
    hashtag = hashtag.lstrip("#")
    url = f"https://www.tiktok.com/tag/{hashtag}"
    collected = []
    all_urls_seen = []  # ← log every response URL

    async def handler(response):
        u = response.url
        all_urls_seen.append(u)

        # Print anything that looks API-related
        if any(x in u for x in ["api", "item_list", "challenge", "feed", "recommend"]):
            print(f"  🔍 API hit: {u[:120]}")
            try:
                body = await response.json()
                print(f"      keys: {list(body.keys())[:8]}")
                # Dump raw so we can see the actual shape
                items = body.get("itemList") or body.get("items") or []
                print(f"      items found: {len(items)}")
                if items:
                    print(f"      first item keys: {list(items[0].keys())[:10]}")
                    collected.extend(items[:max_posts - len(collected)])
            except Exception as e:
                print(f"      (not JSON or parse error: {e})")

    async with async_playwright() as pw:
        browser = await pw.firefox.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"],
        )
        context = await browser.new_context(
            viewport={"width": 1280, "height": 900},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) "
                "Gecko/20100101 Firefox/125.0"
            ),
            locale="en-US",
            extra_http_headers={
                "Accept-Language": "en-US,en;q=0.9",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            },
        )

        # Mask automation flags
        await context.add_init_script("""
            Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
            Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3]});
        """)

        page = await context.new_page()
        page.on("response", handler)

        print(f"Navigating to {url} ...")
        await page.goto(url, wait_until="domcontentloaded", timeout=30_000)
        await page.wait_for_timeout(4000)

        # Check what actually rendered
        title = await page.title()
        print(f"Page title: {title}")

        for i in range(15):
            await page.evaluate("window.scrollBy(0, window.innerHeight * 3)")
            await page.wait_for_timeout(2000)

        print(f"\n--- All response URLs ({len(all_urls_seen)} total) ---")
        for u in all_urls_seen:
            if "tiktok" in u:
                print(" ", u[:120])

        await browser.close()

    print(f"\nCollected: {len(collected)} raw items")
    return collected

items = await scrape_tiktok_debug(HASHTAG, MAX_POSTS)

#This will show us every URL TikTok is actually calling. Once we see what endpoints are firing (or if it's getting a bot-detection page instead), we can fix the actual extractor to match. Paste the output here.Navigating to https://www.tiktok.com/tag/oscarssowhite ...
  #🔍 API hit: https://lf16-tiktok-web.tiktokcdn-us.com/obj/tiktok-web-tx/tiktok/webapp/main/react-v18/webapp-desktop/static/js/async/c
     # (not JSON or parse error: Response.json: Protocol error (Network.getResponseBody): Component returnepasted11:10 AMGood news and bad news. The good news: TikTok is definitely hitting /api/challenge/item_list/ and returning data (you can see videos loading — thumbnails, video files, etc.). The bad news: every item_list response throws NS_ERROR_FAILURE when we try to read the body. This is because TikTok encrypts/compresses those responses in a way Playwright's Firefox can't decode via response.json().
#The fix: scrape the data that's already embedded in the page HTML (TikTok server-side renders the initial batch into a <script> tag), then extract from the DOM for subsequent posts:
import asyncio, json, re, pandas as pd
from datetime import datetime
from playwright.async_api import async_playwright

HASHTAG   = "oscarssowhite"
MAX_POSTS = 500

async def scrape_tiktok(hashtag, max_posts=50):
    hashtag  = hashtag.lstrip("#")
    url      = f"https://www.tiktok.com/tag/{hashtag}"
    collected = []
    seen_ids  = set()

    def extract_from_script(html):
        """TikTok embeds __UNIVERSAL_DATA_FOR_REHYDRATION__ in a script tag."""
        match = re.search(r'<script id="__UNIVERSAL_DATA_FOR_REHYDRATION__"[^>]*>(.*?)</script>', html, re.DOTALL)
        if not match:
            return []
        try:
            data = json.loads(match.group(1))
            # Walk the nested structure to find itemList
            default_scope = data.get("__DEFAULT_SCOPE__", {})
            for key, val in default_scope.items():
                if "itemList" in str(key) or "challenge" in str(key).lower():
                    items = val.get("itemList") or val.get("items") or []
                    if items:
                        return items
            # Brute-force search for itemList anywhere in the dict
            raw = json.dumps(data)
            matches = re.findall(r'"itemList"\s*:\s*(\[.*?\])\s*,\s*"', raw)
            for m in matches:
                try:
                    return json.loads(m)
                except:
                    pass
        except Exception as e:
            print(f"  Script parse error: {e}")
        return []

    def extract_from_dom_nodes(nodes_json_list):
        rows = []
        for item in nodes_json_list:
            try:
                author = item.get("author", {})
                stats  = item.get("stats", {})
                rows.append({
                    "post_id":          item.get("id"),
                    "description":      item.get("desc", "")[:300],
                    "author_username":  author.get("uniqueId"),
                    "author_nickname":  author.get("nickname"),
                    "author_verified":  author.get("verified", False),
                    "likes":            stats.get("diggCount"),
                    "comments":         stats.get("commentCount"),
                    "shares":           stats.get("shareCount"),
                    "views":            stats.get("playCount"),
                    "created_at":       datetime.utcfromtimestamp(item.get("createTime",0)).strftime("%Y-%m-%d %H:%M:%S"),
                    "post_url":         f"https://www.tiktok.com/@{author.get('uniqueId')}/video/{item.get('id')}",
                })
            except:
                pass
        return rows

    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)
        context = await browser.new_context(
            viewport={"width": 1280, "height": 900},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0",
            locale="en-US",
        )
        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"
        )
        page = await context.new_page()
        await page.goto(url, wait_until="networkidle", timeout=30_000)
        await page.wait_for_timeout(3000)

        # ── Pass 1: Extract from embedded script tag ──────────────────────────
        html  = await page.content()
        items = extract_from_script(html)
        print(f"  Script tag items found: {len(items)}")
        for row in extract_from_dom_nodes(items):
            if row["post_id"] not in seen_ids:
                seen_ids.add(row["post_id"])
                collected.append(row)

        # ── Pass 2: Extract from rendered DOM video cards ─────────────────────
        # TikTok renders data-e2e attributes and JSON in article elements
        for scroll_i in range(40):
            if len(collected) >= max_posts:
                break

            # Pull JSON data from each video card's data attribute
            cards = await page.evaluate("""
                () => {
                    const results = [];
                    // Method A: data attributes on article/div cards
                    document.querySelectorAll('[data-e2e="challenge-item"]').forEach(el => {
                        const a = el.querySelector('a[href*="/video/"]');
                        const likeEl = el.querySelector('[data-e2e="like-count"]');
                        const userEl = el.querySelector('[data-e2e="challenge-item-username"]') 
                                    || el.querySelector('a[href*="/@"]');
                        if (a) {
                            results.push({
                                url:   a.href,
                                likes: likeEl ? likeEl.innerText : null,
                                user:  userEl ? (userEl.innerText || userEl.href) : null,
                            });
                        }
                    });
                    // Method B: grab from Next.js / React fiber data
                    const scripts = Array.from(document.querySelectorAll('script[type="application/json"]'));
                    return {cards: results, scriptCount: scripts.length};
                }
            """)

            new_dom = cards.get("cards", [])
            before  = len(collected)
            for c in new_dom:
                vid_url = c.get("url", "")
                vid_id  = re.search(r'/video/(\d+)', vid_url)
                if vid_id:
                    vid_id = vid_id.group(1)
                    if vid_id not in seen_ids:
                        seen_ids.add(vid_id)
                        collected.append({
                            "post_id":    vid_id,
                            "post_url":   vid_url,
                            "likes":      c.get("likes"),
                            "author_username": c.get("user"),
                            "description": None,
                            "views": None,
                        })

            if len(collected) > before:
                print(f"  Scroll {scroll_i+1}: +{len(collected)-before} posts (total {len(collected)})")

            await page.evaluate("window.scrollBy(0, window.innerHeight * 3)")
            await page.wait_for_timeout(2000)

        await browser.close()

    print(f"\n✅ {len(collected)} posts collected")
    return pd.DataFrame(collected)

df = await scrape_tiktok(HASHTAG, MAX_POSTS)

if not df.empty:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"tiktok_{HASHTAG}_{ts}.csv"
    df.to_csv(path, index=False)
    print(f"💾 Saved → {path}")
    print(df.head(10))
else:
    print("⚠️ No posts collected")

Navigating to https://www.tiktok.com/tag/oscarssowhite ...
  🔍 API hit: https://lf16-tiktok-web.tiktokcdn-us.com/obj/tiktok-web-tx/tiktok/webapp/main/react-v18/webapp-desktop/static/js/async/c
      (not JSON or parse error: Response.json: Protocol error (Network.getResponseBody): Component returned failure code: 0x80004005 (NS_ERROR_FAILURE) [nsIStreamListener.onDataAvailable])
  🔍 API hit: https://www.tiktok.com/tiktok/ppf/api/eligibility/v2?WebIdLastTime=1776093113&aid=1988&app_language=en&app_name=tiktok_w
      (not JSON or parse error: Response.json: Protocol error (Network.getResponseBody): Component returned failure code: 0x80004005 (NS_ERROR_FAILURE) [nsIStreamListener.onDataAvailable])
  🔍 API hit: https://www.tiktok.com/api/challenge/detail/?WebIdLastTime=1776093113&aid=1988&app_language=en&app_name=tiktok_web&brows
      (not JSON or parse error: Response.json: Protocol error (Network.getResponseBody): Component returned failure code: 0x80004005 (NS_ERROR_FAILURE) [nsIStrea